# Notebook 3 - Chaîne LangChain complète

### Objectif
- Créer une chaîne LangChain RAG complète
- Générer des réponses juridiques avec le LLM
- Ajouter une mémoire conversationnelle
- Simuler un vrai dialogue juridique

### Prérequis
Avoir exécuté les notebooks 01 et 02

In [1]:
!pip install langchain langchain-openai langchain-community pypdf chromadb python-dotenv -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 334.5/334.5 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.0/23.0 MB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 

## Étape 1 - Configuration et chargement du vectorstore

On charge l'index ChromaDB déjà créé dans le notebook 02.
Pas besoin de re-vectoriser c'est déjà fait

In [7]:
import os
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

DOCS_DIR = "/content/drive/MyDrive/GEN_AI_Assistant_Juridique/data"
CHROMA_LOCAL = "/content/chroma_db"  # chemin dans Colab directement

# Chargement des PDFs
pdf_files = list(Path(DOCS_DIR).glob("*.pdf"))
docs = []
for pdf_path in pdf_files:
    loader = PyPDFLoader(str(pdf_path))
    docs.extend(loader.load())
print(f"{len(docs)} pages chargées")

# Découpage
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=["\nArticle", "\n\n", "\n", " "]
)
splits = splitter.split_documents(docs)
print(f"{len(splits)} chunks créés")

# Vectorisation
print("Vectorisation en cours...")
embedding = OpenAIEmbeddings()
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embedding,
    persist_directory=CHROMA_LOCAL
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

print(f"Vectorstore créé")
print(f"Chunks disponibles : {vectorstore._collection.count()}")

479 pages chargées
2183 chunks créés
Vectorisation en cours...
Vectorstore créé
Chunks disponibles : 2183


## Étape 2 - Création du prompt juridique

On crée un prompt spécialisé qui demande au LLM de :
- Répondre uniquement en se basant sur le Code du Travail
- Citer les articles pertinents (ex: Article L1234-1)
- Répondre en français de manière claire et structurée


In [8]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

prompt = ChatPromptTemplate.from_messages([
    ("system", """Tu es un assistant juridique expert en droit du travail français.
Réponds à la question en te basant UNIQUEMENT sur les extraits
du Code du Travail fournis.
Si l'information n'est pas dans les extraits, dis-le clairement.
Cite les articles pertinents quand possible.
Réponds toujours en français de manière claire et structurée.

Extraits du Code du Travail :
{context}"""),
    ("human", "{question}")
])

print("Prompt juridique créé")

Prompt juridique créé


## Étape 3 - Création de la chaîne LangChain RAG

La chaîne RAG fonctionne en 3 étapes :
1. Retrieval : on récupère les 4 chunks les plus pertinents
2. Augmentation : on injecte ces chunks dans le prompt
3. Generation : le LLM génère une réponse basée sur ces chunks

In [9]:
def rag_chain(question):
    # Étape 1 : Retrieval
    docs = retriever.invoke(question)

    # Étape 2 : Augmentation
    context = "\n\n".join([d.page_content for d in docs])

    # Étape 3 : Generation
    chain = prompt | llm | StrOutputParser()
    reponse = chain.invoke({
        "context": context,
        "question": question
    })

    return reponse, docs

print("Chaîne RAG créée avec succès")

Chaîne RAG créée avec succès


## Étape 4 - Test de la chaîne RAG

On teste avec des questions juridiques variées.
Pour chaque question on affiche :
- La réponse générée par le LLM
- Les sources utilisées (fichier + page)

In [10]:
questions_test = [
    "Quel est le délai de préavis pour un CDI ?",
    "Quelles sont les conditions du licenciement pour faute grave ?",
    "Quels sont les droits aux congés payés ?",
    "Quelles sont les règles sur le harcèlement moral ?",
    "Comment fonctionne la rupture conventionnelle ?"
]

for question in questions_test:
    print(f"\n{'='*60}")
    print(f"Question : {question}")
    print(f"{'='*60}")
    reponse, sources = rag_chain(question)
    print(f"\nRéponse :\n{reponse}")
    print(f"\nSources utilisées :")
    for doc in sources:
        fichier = doc.metadata['source'].split('/')[-1]
        page = doc.metadata.get('page', '?')
        print(f"  - {fichier} (page {page})")


Question : Quel est le délai de préavis pour un CDI ?

Réponse :
Le délai de préavis pour un CDI est d'un jour par semaine, en fonction de la durée totale du contrat incluant les renouvellements éventuels, lorsque celui-ci comporte un terme précis, ou de la durée effectuée lorsque le contrat n'a pas de terme précis. Ce délai ne peut pas être inférieur à un jour ni supérieur à deux semaines (Code du Travail, p.148).

Sources utilisées :
  - Contrat_de_travail.pdf (page 105)
  - Contrat_de_travail.pdf (page 122)
  - Contrat_de_travail.pdf (page 5)
  - Contrat_de_travail.pdf (page 7)

Question : Quelles sont les conditions du licenciement pour faute grave ?

Réponse :
Les conditions du licenciement pour faute grave sont les suivantes, selon les extraits du Code du Travail fournis :

- En cas de faute grave, l'employeur peut prononcer la mise à pied immédiate de l'intéressé dans l'attente de la décision définitive. Si le licenciement pour faute grave est refusé, la mise à pied est annulée